# Experiment 01 — Optuna dev-HPO run (train vs validation loss)

Question: which hyperparameters (lr, dropout, burn-in warmup, LR-decay point/factor,
batch size, grad accumulation) give the best validation loss for the tiny-tab T5
on the 5% dev split (`dev-hpo-split-v1`: 68,794 train / 8,864 val windows)?

Design: each trial trains at most 5 epochs (45-min wall-clock backstop), LR schedule =
linear burn-in over `warmup_examples` → hold → single multiplicative decay at
`decay_after_examples`. Data source: per-trial `trial_<n>_history.json` under
`models/hpo-dev-v1/` plus the `hpo-dev.db` study database (copied to /tmp before
reading, so this notebook never locks the running study).

**Refresh by re-running all cells.**

In [ ]:
import glob
import json
import shutil
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import optuna
import pandas as pd

%matplotlib inline

REPO = Path.cwd()
while REPO != REPO.parent and not (REPO / "hpo-dev.db").exists():
    REPO = REPO.parent
OUT_ROOT = REPO / "models" / "hpo-dev-v1"
print("repo:", REPO)

histories = {}
for f in sorted(glob.glob(str(OUT_ROOT / "trial_*" / "trial_*_history.json"))):
    h = json.loads(Path(f).read_text())
    histories[h["trial"]] = h
print(f"trials with curve data: {sorted(histories)}")

DB_COPY = Path(tempfile.gettempdir()) / "hpo-exp01-readcopy.db"
study = None
if (REPO / "hpo-dev.db").exists():
    shutil.copy2(REPO / "hpo-dev.db", DB_COPY)
    for name in ("dev-hpo-v4", "dev-hpo-v3", "dev-hpo-v2"):
        try:
            study = optuna.load_study(study_name=name, storage=f"sqlite:///{DB_COPY}")
            print(f"loaded study '{name}': {len(study.trials)} trials")
            break
        except KeyError:
            continue
if study is None:
    print("no study DB found — showing history-JSON data only")

In [ ]:
# --- Train loss curves (all trials) ---
fig, ax = plt.subplots(figsize=(9, 4))
for tid in sorted(histories):
    tr = histories[tid]["train_loss_curve"]
    if tr:
        ax.plot([p["step"] for p in tr], [p["loss"] for p in tr],
                alpha=0.8, label=f"trial {tid}")
ax.set_xlabel("optimizer step"); ax.set_ylabel("train loss")
ax.set_title("Train loss per trial")
ax.legend(fontsize=8)
plt.show() if histories else print("no train curves on disk yet")

In [ ]:
# --- Validation loss curves (all trials) ---
fig, ax = plt.subplots(figsize=(9, 4))
for tid in sorted(histories):
    ev = histories[tid]["eval_loss_curve"]
    if ev:
        ax.plot([p["step"] for p in ev], [p["eval_loss"] for p in ev],
                "o-", ms=3, alpha=0.85, label=f"trial {tid}")
ax.set_xlabel("optimizer step"); ax.set_ylabel("eval loss")
ax.set_title("Validation loss per trial (held-out dev val split)")
ax.legend(fontsize=8)
plt.show() if histories else print("no eval curves on disk yet")

In [ ]:
# --- Train vs validation overlay: best trial by final val_loss ---
ranked = sorted(
    ((tid, h["val_loss"]) for tid, h in histories.items()
     if h.get("eval_loss_curve")), key=lambda kv: kv[1])
if ranked:
    tid, best = ranked[0]
    h = histories[tid]
    fig, ax = plt.subplots(figsize=(9, 4))
    tr = h["train_loss_curve"]
    ev = h["eval_loss_curve"]
    ax.plot([p["step"] for p in tr], [p["loss"] for p in tr],
            alpha=0.7, label="train loss")
    ax.plot([p["step"] for p in ev], [p["eval_loss"] for p in ev],
            "o-", ms=4, label="val loss")
    ax.set_xlabel("optimizer step"); ax.set_ylabel("loss")
    ax.set_title(f"Best trial so far: {tid} (val {best:.3f}) — "
                  f"batch {h['params']['batch_size']}x{h['params']['grad_accum']}, "
                  f"lr {h['params']['lr']:.1e}")
    ax.legend()
    plt.show()
else:
    print("no completed eval curves yet")

In [ ]:
# --- Summary table: first/last train loss, best val, config ---
rows = []
for tid in sorted(histories):
    h = histories[tid]
    tr = h["train_loss_curve"]
    ev = h["eval_loss_curve"]
    rows.append({
        "trial": tid,
        "train_first": round(tr[0]["loss"], 3) if tr else None,
        "train_last": round(tr[-1]["loss"], 3) if tr else None,
        "val_best": round(min(p["eval_loss"] for p in ev), 3) if ev else None,
        "val_last": round(ev[-1]["eval_loss"], 3) if ev else None,
        "batch": f"{h['params']['batch_size']}x{h['params']['grad_accum']}",
        "lr": f"{h['params']['lr']:.1e}",
        "warmup": h["params"]["warmup_examples"],
        "decay@": h["params"]["decay_after_examples"],
        "decay_x": h["params"]["decay_factor"],
    })
pd.DataFrame(rows).sort_values("val_best") if rows else print("no data yet")